# Chapter 6: Decorators

## Writing Our First Decorator


In [1]:
def say_hello(name):
    """This function says hello"""
    print(f"Hello, {name}")

In [2]:
print(say_hello.__name__)
print(say_hello.__doc__)

say_hello
This function says hello


In [3]:
def decorator(func):
    print("decorator executed")

    def wrapper(*args, **kwargs):
        "This function is the wrapper"
        print("wrapper executed")
        return func(*args, **kwargs)

    return wrapper

In [4]:
@decorator
def say_hello(name):
    """This function says hello"""
    print(f"Hello, {name}")


# equivalent to:

# def say_hello(name):
#     """This function says hello"""
#     print(f"Hello, {name}")

# wrapper = decorator(say_hello)

decorator executed


In [5]:
say_hello("Bob")

wrapper executed
Hello, Bob


In [6]:
print(say_hello.__name__)
print(say_hello.__doc__)

wrapper
This function is the wrapper


## Preserving Metadata with `functools.wraps`

In [7]:
from functools import wraps

In [8]:
def decorator(func):
    print("decorator executed")

    @wraps(func)
    def wrapper(*args, **kwargs):
        "This function is the wrapper"
        print("wrapper executed")
        return func(*args, **kwargs)

    return wrapper

In [9]:
@decorator
def say_hello(name):
    """This function says hello"""
    print(f"Hello, {name}")

decorator executed


In [10]:
say_hello("Bob")

wrapper executed
Hello, Bob


In [11]:
print(say_hello.__name__)
print(say_hello.__doc__)

say_hello
This function says hello


In [12]:
import time


def timer(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"{func.__name__} ran in {end-start:.6f} seconds")
        return result

    return wrapper

In [13]:
@timer
def slow_run(n):
    return sum(range(n))

In [14]:
slow_run(20_000_000)

slow_run ran in 0.182492 seconds


199999990000000

In [15]:
print(slow_run.__name__)

slow_run


## Decorator with Arguments

In [16]:
def repeat(n):

    print("repeat executed")

    def external_wrapper(func):

        print("external_wrapper executed")

        @wraps(func)
        def wrapper(*args, **kwargs):

            print("wrapper executed")

            for _ in range(n):
                func(*args, **kwargs)

        return wrapper

    return external_wrapper

In [17]:
@repeat(5)
def say_hello(name):
    """This function says hello"""
    print(f"Hello, {name}")


# equivalent to:
# def say_hello(name):
#     """This function says hello"""
#     print(f"Hello, {name}")


# say_hello = repeat(2)(say_hello)

repeat executed
external_wrapper executed


In [18]:
say_hello("Alice")

wrapper executed
Hello, Alice
Hello, Alice
Hello, Alice
Hello, Alice
Hello, Alice


In [19]:
say_hello.__name__

'say_hello'

## Stacked Decorators

In [20]:
def decorator1(func):
    print("decorator1 executed")

    @wraps(func)
    def wrapper1(*args, **kwargs):
        print("wrapper1 executed")
        return func(*args, **kwargs)

    return wrapper1


def decorator2(func):
    print("decorator2 executed")

    @wraps(func)
    def wrapper2(*args, **kwargs):
        print("wrapper2 executed")
        return func(*args, **kwargs)

    return wrapper2

In [21]:
@decorator2
@decorator1
def say_hello(name):
    """This function says hello"""
    print(f"Hello, {name}")


# equivalent to

# def say_hello(name):
#     """This function says hello"""
#     print(f"Hello, {name}")

# say_hello = decorator2(decorator1(say_hello))

decorator1 executed
decorator2 executed


In [22]:
say_hello("Richard")

wrapper2 executed
wrapper1 executed
Hello, Richard


In [23]:
say_hello.__name__

'say_hello'

## `functools.cache`

In [24]:
from functools import cache
from time import sleep


@cache
def factorial(n):
    sleep(0.5)
    if n < 2:
        return 1
    return n * factorial(n - 1)

In [25]:
start = time.perf_counter()
factorial(8)
end = time.perf_counter()
print(f"First call took {end - start:.6f} seconds")

First call took 4.004061 seconds


In [26]:
start = time.perf_counter()
factorial(4)
end = time.perf_counter()
print(f"Second call took {end - start:.6f} seconds")

Second call took 0.000038 seconds


## `functools.partial`

In [27]:
from functools import partial


def power(base, exponent):
    return base**exponent

In [28]:
square = partial(power, exponent=2)
cube = partial(power, exponent=3)

In [29]:
print(square(4))
print(cube(4))

16
64


## `functools.singledispatch`

In [30]:
from functools import singledispatch


@singledispatch
def describe(obj):
    return f"Object of type {type(obj).__name__}"


@describe.register(int)
def _(n):
    return f"This is an integer! The integer is {n}"


@describe.register(float)
def _(f):
    return f"This is a float! The float is {f}"


@describe.register(tuple)
def _(t):
    return f"This is a tuple! The tuple is {t}"

In [31]:
describe(4)

'This is an integer! The integer is 4'

In [32]:
describe(4.0)

'This is a float! The float is 4.0'

In [33]:
describe((4,))

'This is a tuple! The tuple is (4,)'